# Basic API

The code below shows how to use Knwler as an API. You can take out the pipeline parts you need in your context.
Note that the methods shown below are wrappers in order to make it as easy as possible to perform various tasks via a single namespace, i.e. the `knwler.api` import. You can import the underlying methods as well but are usually somewhat more involved.

This notebook is using Knwler v1.0.6 or above.



## Install Knwler

There are [various ways you can install knwler](https://knwler.com/docs/setup.html), the easiest being to use `uv add knwler` after running `uv init` in a directory. You can ensure it's installed via something like:

In [60]:
import knwler
from importlib.metadata import version
print("knwler v{}".format(version("knwler")))

knwler v1.0.12


## Config
The various API methods accept an optional `Config` instance which in essence defines the LLM and its properties. 

In [61]:
from knwler.api import Config
config = Config(backend="ollama", discovery_model="gemma3:4b", extraction_model="gemma3:12b ")

## Fetching data

The API allows you to fetch an arbitrary url, file or Wikipedia articles.

In [62]:
from knwler.api import fetch_wikipedia_page
await fetch_wikipedia_page("Python (programming language)")

{'text': 'Python is a high-level, general-purpose programming language. Its design philosophy emphasizes code readability with the use of significant indentation. Python is dynamically type-checked and garbage-collected. It supports multiple programming paradigms, including structured (particularly procedural), object-oriented and functional programming.\nGuido van Rossum began working on Python in the late 1980s as a successor to the ABC programming language. Python 3.0, released in 2008, was a major revision and not completely backward-compatible with earlier versions. Beginning with Python 3.5, capabilities and keywords for typing were added to the language, allowing optional static typing. As of 2026, the Python Software Foundation supports Python 3.10, 3.11, 3.12, 3.13, and 3.14, following the project\'s annual release cycle and five-year support policy. Python 3.15 is currently in the alpha development phase, and the stable release is expected to come out in October 2026. Earlier

When fetching a web page or a pdf the fetch method will return a tuple `metadata, content`:

In [63]:
from knwler.api import fetch_url
import json
metadata, content = await fetch_url("https://www.wikipedia.org/")
print(json.dumps(metadata, indent=2))

{
  "text": "Wikipedia\n\n\n\n\n![](portal/wikipedia.org/assets/img/Wikipedia-logo-v2.png)\n\nWikipedia\n**The Free Encyclopedia**\n===================================\n\nWikipedia\n**25 years of the free encyclopedia**\n===============================================\n\n[**English**\n7,141,000+ articles](//en.wikipedia.org/ \"English \u2014 Wikipedia \u2014 The Free Encyclopedia\")\n\n[**\u65e5\u672c\u8a9e**\n1,491,000+ \u8a18\u4e8b](//ja.wikipedia.org/ \"Nihongo \u2014 \u30a6\u30a3\u30ad\u30da\u30c7\u30a3\u30a2 \u2014 \u30d5\u30ea\u30fc\u767e\u79d1\u4e8b\u5178\")\n\n[**Deutsch**\n3.099.000+ Artikel](//de.wikipedia.org/ \"Deutsch \u2014 Wikipedia \u2014 Die freie Enzyklop\u00e4die\")\n\n[**\u0420\u0443\u0441\u0441\u043a\u0438\u0439**\n2\u00a0087\u00a0000+ \u0441\u0442\u0430\u0442\u0435\u0439](//ru.wikipedia.org/ \"Russkiy \u2014 \u0412\u0438\u043a\u0438\u043f\u0435\u0434\u0438\u044f \u2014 \u0421\u0432\u043e\u0431\u043e\u0434\u043d\u0430\u044f \u044d\u043d\u0446\u0438\u043a\u043b\u043

In [64]:
metadata, content = await fetch_url("https://knwler.com/pdfs/HumanRights.pdf")
print(json.dumps(metadata, indent=2))

{
  "id": "https://knwler.com/pdfs/HumanRights.pdf",
  "name": "HumanRights.pdf",
  "file_path": "/Users/swa/.knwler/cache/documents/7cc2d8fb74f70bdb82d194fb5ddc59c55f238dbf17dfed0375034b34495141c2.pdf",
  "extension": "pdf",
  "cached": true
}


and the content will be in this case binary. You can save it like so:

In [65]:
with open("HumanRights.pdf", "wb") as f:
    f.write(content)

## Parsing PDF files

In [66]:
from knwler.api import parse_pdf
import os

md = await parse_pdf(os.path.join(os.getcwd(), "HumanRights.pdf"))
print(md[:500])

Universal Declaration of Human Rights 
Preamble 
Whereas recognition of the inherent dignity and of the equal and inalienable 
rights of all members of the human family is the foundation of freedom, justice 
and peace in the world,  
Whereas disregard and contempt for human rights have resulted in barbarous 
acts which have outraged the conscience of mankind, and the advent of a world 
in which human beings shall enjoy freedom of speech and belief and freedom 
from fear and want has been proclai


## Chunking

You can use the methods above to fetch markdown or you have some text/markdown available:

In [67]:
# use some pdf 
md = await parse_pdf(os.path.join(os.getcwd(), "HumanRights.pdf"))

 # or read in text
# with open("~/document.md", "r") as f:
#     text = f.read()
from knwler.api import Config, chunk
config = Config(max_tokens=200, overlap_tokens=20)   
chunks = await chunk(md, config=config)


In [68]:
print(f"There are {len(chunks)} chunks, the first chunk is:\n\n{chunks[0]}")

There are 14 chunks, the first chunk is:

Chunk(text='Universal Declaration of Human Rights \nPreamble \nWhereas recognition of the inherent dignity and of the equal and inalienable \nrights of all members of the human family is the foundation of freedom, justice \nand peace in the world,  \nWhereas disregard and contempt for human rights have resulted in barbarous \nacts which have outraged the conscience of mankind, and the advent of a world \nin which human beings shall enjoy freedom of speech and belief and freedom \nfrom fear and want has been proclaimed as the highest aspiration of the common \npeople,  \nWhereas it is essential, if man is not to be compelled to have recourse, as a last \nresort, to rebellion against tyranny and oppression, that human rights should be \nprotected by the rule of law,  \nWhereas it is essential to promote the development of friendly relations between \nnations,  \nWhereas the peoples of the United Nations have in the Charter reaffirmed their \nfait

## Schema

Simply said, a schema defines the type of words and the relations between these words that matter to you. It is a set of entity types and relationships. You can define them or you can let Knwler decide (via a LLM):

In [69]:
from knwler.api import infer_schema
from dataclasses import asdict
import json
schema = await infer_schema(md, config=config)
print(json.dumps(asdict(schema), indent=2))

{
  "entity_types": [
    "declaration",
    "article",
    "right",
    "freedom",
    "principle",
    "nation",
    "state",
    "person",
    "family",
    "property",
    "religion",
    "marriage",
    "asylum",
    "nationality",
    "conscience",
    "thought",
    "speech",
    "art",
    "literature",
    "scientific_production"
  ],
  "relation_types": [
    "declares",
    "defines",
    "protects",
    "guarantees",
    "violates",
    "promotes",
    "requires",
    "supports",
    "opposes",
    "limits",
    "includes",
    "concerns",
    "entitles_to",
    "prohibits",
    "allows",
    "regulates",
    "protects_from",
    "derives_from",
    "results_in",
    "affects"
  ],
  "reasoning": "The text primarily focuses on the Universal Declaration of Human Rights, detailing various rights and freedoms. It outlines principles that nations should uphold to ensure these rights are protected and respected. The entity types cover key concepts mentioned in the declaration, w

The inferred schema can be changed after discovery, there is nothing holy in what the LLM suggests. If you want to define your own schema you can use something like this:

In [70]:
from knwler.models import Schema
schema = Schema(entity_types=["person", "size"], relation_types=["has_size"])

## Language

The graph extraction uses internally language detection but if you want to use it standalone you can access it like this:

In [71]:
from knwler.api import discover_language
dic = {
    "fr": "L'importance de la découverte de la langue ne peut être surestimée.",
    "es": "La importancia del descubrimiento del idioma no puede ser subestimada.",
    "cn":"语言发现的重要性不容小觑。",
    "nl": "Het belang van taalontdekking kan niet worden overschat.",
    "en": "The importance of language discovery cannot be overstated."
    
}
for lang, sentence in dic.items():
    language = await discover_language(sentence, config=config)
    print(f"Detected language for '{sentence}': {language}")    


Detected language for 'L'importance de la découverte de la langue ne peut être surestimée.': fr
Detected language for 'La importancia del descubrimiento del idioma no puede ser subestimada.': es
Detected language for '语言发现的重要性不容小觑。': zh
Detected language for 'Het belang van taalontdekking kan niet worden overschat.': nl
Detected language for 'The importance of language discovery cannot be overstated.': en


If you want to set/override the language use the `set_language` method from `knwler.api`.

## Extraction

You can extract a pdf, a piece of text of a collection of chunks with the same API method.

In [72]:
from knwler.api import extract, Config, Schema
from knwler.language import get_current_language
schema = Schema(entity_types=["animal", "object"], relation_types=["is_on"])
g = await extract("The cat is on the table.", schema, config=Config())

Output()

The result contains the schema, the knowledge graph and the chunks:

In [73]:
import json
from dataclasses import asdict
print(json.dumps(asdict(g), indent=2))

{
  "graph": {
    "entities": [
      {
        "name": "cat",
        "type": "animal",
        "description": "A feline pet.",
        "chunk_ids": [
          "f880d214-f3c8-4751-be3f-129cb319a49c"
        ],
        "id": "28638096-e8cb-489f-9b4f-fe5a48e77f29"
      },
      {
        "name": "table",
        "type": "object",
        "description": "A piece of furniture used for eating, writing, or working on.",
        "chunk_ids": [
          "f880d214-f3c8-4751-be3f-129cb319a49c"
        ],
        "id": "fd52ca55-e001-4926-b3dc-de79e717a1cd"
      }
    ],
    "relations": [
      {
        "source": "cat",
        "source_type": "animal",
        "target": "table",
        "target_type": "object",
        "type": "is_on",
        "description": "The cat is positioned above the surface of the table.",
        "strength": 8.0,
        "chunk_ids": [
          "f880d214-f3c8-4751-be3f-129cb319a49c"
        ],
        "id": "27ac8b4e-7937-4620-85e9-d86e8ec333ae"
      }
    ]
  

You can also give the method a pdf. In this short pdf there are nine people coming from various countries:

In [74]:
from knwler.api import Chunk, Schema, Config, extract
from pathlib import Path
import os
file_path = Path("./People.pdf")
if not file_path.exists():
    from knwler.api import fetch_url
    url = "https://knwler.com/pdfs/People.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path, "wb") as f:
        f.write(bytes)
schema = Schema(entity_types=["person", "country"], relation_types=["is_from"])
r = await extract(file_path, schema=schema, config=Config())
print([e["name"] for e in r.graph.entities if e["type"] == "person"])


Output()

['Elena Petrova', 'Samuel Okoye', 'Lucía Fernández', 'Noah Williams', 'Sofia Dimitriou', 'Hassan Al-Masri', 'Mei-Ling Chen', 'Arjun Mehta', 'Oliver Grant']


## Consolidation

If you have various documents around a common domain you likely want to merge the graphs extracted from these documents. 

In [75]:
from knwler.api import *
schema1 = Schema(entity_types=["person", "location"], relation_types=["is_from"])
file_path1 = Path("./People.pdf")
if not file_path1.exists():
    url = "https://knwler.com/pdfs/People.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path1, "wb") as f:
        f.write(bytes)

schema2 = Schema(entity_types=["city", "country"], relation_types=["located_in"])
file_path2 = Path("./Places.pdf")
if not file_path2.exists():
    url = "https://knwler.com/pdfs/Places.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path2, "wb") as f:
        f.write(bytes)
config = Config(extraction_model="gemma3:12b")
g1 = await extract(file_path1, schema=schema1, config=config)
g2 = await extract(file_path2, schema=schema2, config=config)

g = await consolidate_document_graphs([g1, g2], config=config, include_chunks=True)
print(json.dumps(asdict(g), indent=2))

Output()

Output()

✓ Graph c71e6678-f168-4305-9fe7-71afff9d9278: 1 chunk(s), 18 entities, 9 relations.

✓ Graph 5954b384-cef4-4533-9aaf-84540438854f: 1 chunk(s), 18 entities, 9 relations.

✓ Aggregated 2 chunks, 2 documents, 4 entity types, 2 relation types from all graphs.

{
  "graph": {
    "entities": [
      {
        "name": "Elena Petrova",
        "type": "person",
        "description": "Elena Petrova is a member of the international research collective. She is from Russia.",
        "chunk_ids": [
          "caf00d32-0629-4b02-8355-c7aec9fed598"
        ],
        "id": "904bfe91-2004-4d06-a471-c0862617de4b"
      },
      {
        "name": "Samuel Okoye",
        "type": "person",
        "description": "Samuel Okoye is a member of the international research collective. He is from Nigeria.",
        "chunk_ids": [
          "caf00d32-0629-4b02-8355-c7aec9fed598"
        ],
        "id": "2b30b1e8-e217-4c62-a172-1d257cb48efe"
      },
      {
        "name": "Luc\u00eda Fern\u00e1ndez",
        "type": "person",
        "description": "Luc\u00eda Fern\u00e1ndez is a field ecologist and member of the international research collective. She is from Chile.",
        "chunk_ids": [
          "caf00d32-0629-4b02-8355-c7aec9fed598"
        ],
        "i

## Extras

There are some extra methods in the API you might find useful.

Rephrasing chunks can be useful for reporting but you can also use it prior to extracting graph, this is usually called chunk compression.

In [76]:
from knwler.api import *
chunks = [Chunk(id="chunk1", text="> The $ cat is on the table."), Chunk(id="chunk2", text="The >>dog is in the garden.")]
rephrased_chunks = await rephrase_chunks(chunks, config=Config(extraction_model="gemma3:12b"))
for original, rephrased in zip(chunks, rephrased_chunks):
    print(f"Original: {original.text}\nRephrased: {rephrased}\n")

Output()

Original: > The $ cat is on the table.
Rephrased: Chunk(text='The text says: "The cat is on the table."', chunk_idx=0, id='chunk1')

Original: The >>dog is in the garden.
Rephrased: Chunk(text='A dog is in the garden.', chunk_idx=0, id='chunk2')



The title of a document is extracted from the first few paragraphs:

In [77]:
from knwler.api import *
text ="""
Lovelace's educational and social exploits brought her into contact with scientists such as Andrew Crosse, Charles Babbage, David Brewster, Charles Wheatstone and Michael Faraday, and the author Charles Dickens, contacts which she used to further her education. Lovelace described her approach as "poetical science" and herself as an "Analyst (& Metaphysician)". When she was eighteen, Lovelace's mathematical talents led her to a long working relationship and friendship with fellow British mathematician Charles Babbage. She was particularly interested in Babbage's work on the analytical engine. Lovelace first met him on 5 June 1833, when she and her mother attended one of Charles Babbage's Saturday night soirées with their mutual friend, and Lovelace's private tutor, Mary Somerville.
"""
title = await extract_title(text, config=Config(extraction_model="gemma3:12b"))
print(f"Extracted title: {title}")

Extracted title: Ada Lovelace's Scientific Connections


The summary of text or chunks is also handy for reporting:

In [78]:
from knwler.api import *
file_path = Path("./HumanRights.pdf") 
if not file_path.exists():
    url = "https://knwler.com/pdfs/HumanRights.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path, "wb") as f:
        f.write(bytes)
content = await parse_pdf(file_path)
summary = await extract_summary(content, config=Config(extraction_model="gemma3:12b"))
print(f"Extracted summary: {summary}")        

Extracted summary: This document is the Universal Declaration of Human Rights, a foundational text establishing a common standard for achieving human rights and freedoms for all people and nations. It asserts that all individuals are born free and equal in dignity and rights, without discrimination, and are entitled to fundamental rights such as the right to life, liberty, security, and a fair trial. The declaration emphasizes the importance of protecting these rights through the rule of law, promoting understanding, and preventing recourse to rebellion against oppression, ultimately aiming for a world where everyone enjoys freedom from fear and want.


## Export

With the API methods above you can turn the resulting output to a NetworkX graph, HTML and many other formats.

Let's first recreate a simple graph:

In [79]:
from knwler.api import *
from pathlib import Path
import os
file_path = Path("./People.pdf")
if not file_path.exists():
    from knwler.api import fetch_url
    url = "https://knwler.com/pdfs/People.pdf"
    metadata, bytes = await fetch_url(url, no_cache=True)
    with open(file_path, "wb") as f:
        f.write(bytes)
schema = Schema(entity_types=["person", "country"], relation_types=["is_from"])
doc = await extract(file_path, schema=schema, config=Config())


Output()

Turning this into a NetworkX graph is as simple as:

In [80]:
g = await create_network(doc)
print(f"Graph has {len(g.nodes)} nodes and {len(g.edges)} edges.")

Graph has 17 nodes and 8 edges.


You can use this NetworkX graph in all sorts of ways, notably to apply graph analytics (page rank and centrality in general).
Clustering in Knwler goes a step beyond standard graph analytics since it adds cluster labels to help understand what a cluster is mainly about:

In [81]:
from knwler.clustering import cluster_graph
g = await cluster_graph(doc.graph)
for cluster in g.clusters:
    print(f"Cluster {cluster['id']} with topics {cluster['topics']} contains nodes: {cluster['members']}")

Detected 8 clusters

Cluster 0 with topics ['International Research Collaboration'] contains nodes: ['Elena Petrova::person', 'Samuel Okoye::person']
Cluster 1 with topics ['Biodiversity Data Integration', 'Field Ecology'] contains nodes: ['Elena Petrova, Samuel Okoye::person', 'Lucía Fernández::person']
Cluster 2 with topics ['Data Engineering', 'International Collaboration'] contains nodes: ['Elena Petrova, Samuel Okoye, Lucía Fernández::person', 'Noah Williams::person']
Cluster 3 with topics ['Ethical Implications', 'Predictive Modeling'] contains nodes: ['Elena Petrova, Samuel Okoye, Lucía Fernández, Noah Williams::person', 'Sofia Dimitriou::person']
Cluster 4 with topics ['Computational Sustainability', 'International Research'] contains nodes: ['Elena Petrova, Samuel Okoye, Lucía Fernández, Noah Williams, Sofia Dimitriou, Hassan Al-Masri::person', 'Mei-Ling Chen::person']
Cluster 5 with topics ['Cross-Regional Data Analysis', 'Sustainability'] contains nodes: ['Arjun Mehta::person', 'Mei-Ling Chen, E

Finally, if you want to export a document graph to HTML you can assemble things into a dictionary and hand it over to the renderer.

In [82]:
html = await render_html(asdict(doc))

In [83]:
with open("output.html", "w") as f:
    f.write(html)